### 1.3.5.10. Second-Order Taylor Expansion

$$
f(\mathbf{x}) \approx f(\mathbf{x}_0)
+ \nabla f(\mathbf{x}_0)^{\top}(\mathbf{x} - \mathbf{x}_0)
+ \tfrac{1}{2}(\mathbf{x} - \mathbf{x}_0)^{\top} H(\mathbf{x}_0)\,(\mathbf{x} - \mathbf{x}_0) .
$$

**Explanation:**

The second-order Taylor expansion is the best local quadratic model of a scalar field: it adds the curvature term built from the [Hessian](./09_hessian_matrix.ipynb) to the [tangent-plane](./06_linearization_and_tangent_plane.ipynb) linear model, generalizing the one-variable [quadratic approximation](../04_Sequences_Series_and_Taylor_Approximation/06_quadratic_approximation_and_curvature.ipynb). The gradient term gives the slope and the quadratic form $\tfrac12\,\Delta\mathbf{x}^\top H\,\Delta\mathbf{x}$ gives the bowl or saddle shape. Minimizing this quadratic model is one [Newton step](../../../03_Optimization/01_Unconstrained_Optimization/14_newtons_method.ipynb), and the model underlies trust-region methods and second-order optimality analysis.

**Properties:**
- The model matches $f$ in value, gradient, and Hessian at $\mathbf{x}_0$, with error $O(\|\mathbf{x}-\mathbf{x}_0\|^3)$.
- The quadratic term is a [quadratic form](../../01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb) in $H(\mathbf{x}_0)$.

**Numerical Example:**

Expand $f(x,y) = e^{x}\cos y$ about $\mathbf{x}_0 = (0, 0)$. There $f(0,0) = 1$,

$$
\nabla f(0,0) = \begin{bmatrix} e^x\cos y \\ -e^x\sin y \end{bmatrix}_{(0,0)} = \begin{bmatrix} 1 \\ 0 \end{bmatrix},
\qquad
H(0,0) = \begin{bmatrix} e^x\cos y & -e^x\sin y \\ -e^x\sin y & -e^x\cos y \end{bmatrix}_{(0,0)} = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}.
$$

So the quadratic model is $q(x,y) = 1 + x + \tfrac12(x^2 - y^2)$. At $(0.1, 0.1)$: $q = 1 + 0.1 + \tfrac12(0.01 - 0.01) = 1.1$, versus the true $e^{0.1}\cos(0.1) = 1.0996\ldots$ — agreement to three decimals.

In [1]:
import sympy as sp

x, y = sp.symbols("x y")
f = sp.exp(x) * sp.cos(y)
variables = sp.Matrix([x, y])
base_point = {x: 0, y: 0}

gradient = sp.Matrix([sp.diff(f, variable) for variable in variables])
hessian = sp.hessian(f, variables)
displacement = variables - sp.Matrix([0, 0])

quadratic_model = (f.subs(base_point)
                   + (gradient.subs(base_point).T * displacement)[0]
                   + sp.Rational(1, 2) * (displacement.T * hessian.subs(base_point) * displacement)[0])

query = {x: sp.Rational(1, 10), y: sp.Rational(1, 10)}
model_value = quadratic_model.subs(query)
true_value = f.subs(query)

print("∇f(0,0) =", gradient.subs(base_point).T)
print("H(0,0)  ="); sp.pprint(hessian.subs(base_point))
print("quadratic model q(x,y) =", sp.expand(quadratic_model))
print("q(0.1,0.1)  =", float(model_value))
print("f(0.1,0.1)  =", float(true_value))

∇f(0,0) = Matrix([[1, 0]])
H(0,0)  =
⎡1  0 ⎤
⎢     ⎥
⎣0  -1⎦
quadratic model q(x,y) = x**2/2 + x - y**2/2 + 1
q(0.1,0.1)  = 1.1
f(0.1,0.1)  = 1.0996496668294091


**Equivalent CasADi implementation:**

CasADi assembles the same quadratic model from the value, gradient, and Hessian at the base point.

In [2]:
import casadi as ca

variables = ca.SX.sym("x", 2)
f = ca.exp(variables[0]) * ca.cos(variables[1])
hessian, gradient = ca.hessian(f, variables)
model = ca.Function("model", [variables], [f, gradient, hessian])

base_point = ca.DM([0, 0])
base_value, base_gradient, base_hessian = model(base_point)
displacement = ca.DM([0.1, 0.1]) - base_point
quadratic_value = base_value + ca.dot(base_gradient, displacement) + 0.5 * ca.bilin(base_hessian, displacement, displacement)

print("q(0.1,0.1) =", float(quadratic_value))
print("f(0.1,0.1) =", float(model(ca.DM([0.1, 0.1]))[0]))

q(0.1,0.1) = 1.1
f(0.1,0.1) = 1.0996496668294093


**References:**

[📘 Strang, G. (2016). *Calculus Volume 3*. OpenStax.](https://openstax.org/details/books/calculus-volume-3)  
[📗 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*, Appendix A.4.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Hessian Matrix](./09_hessian_matrix.ipynb) | [Next: Critical Points and the Second Derivative Test ➡️](./11_critical_points_and_second_derivative_test.ipynb)